In [1]:
%load_ext autoreload
%autoreload 2

In [9]:
import json
from pprint import pprint
from langchain_community.llms.huggingface_text_gen_inference import HuggingFaceTextGenInference
from meeplemate.llm_models import EnhancedHuggingFaceTextGenInference, load_tgi_chat_model, load_tokenizer
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
import requests

In [3]:
!curl tgi:80/generate \
    -X POST \
    -d '{"inputs":"What is Deep Learning?","parameters":{"max_new_tokens":20, "details":true, "seed":1, "do_sample":true, "temperature": 0.4}}' \
    -H 'Content-Type: application/json'

{"generated_text":"\n\nDeep Learning is a subset of Machine Learning that uses neural networks with multiple layers to process and","details":{"finish_reason":"length","generated_tokens":20,"seed":1,"prefill":[],"tokens":[{"id":13,"text":"\n","logprob":-0.0004518032,"special":false},{"id":13,"text":"\n","logprob":-0.0049209595,"special":false},{"id":23229,"text":"Deep","logprob":-0.00025963783,"special":false},{"id":17504,"text":" Learning","logprob":-1.2900391,"special":false},{"id":349,"text":" is","logprob":-0.00092840195,"special":false},{"id":264,"text":" a","logprob":-0.0031051636,"special":false},{"id":19804,"text":" subset","logprob":-0.07147217,"special":false},{"id":302,"text":" of","logprob":0.0,"special":false},{"id":13253,"text":" Machine","logprob":-0.31420898,"special":false},{"id":17504,"text":" Learning","logprob":-0.00028681755,"special":false},{"id":369,"text":" that","logprob":-2.3808594,"special":false},{"id":6098,"text":" uses","logprob":-0.13830566,"special":fals

In [11]:
requests.get('http://tgi:80/info').json()["model_id"]

{'model_id': 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ',
 'model_sha': 'a0e39e3a0ca6767307293819acb7c38e7a18cd31',
 'model_dtype': 'torch.float16',
 'model_device_type': 'cuda',
 'model_pipeline_tag': 'text-generation',
 'max_concurrent_requests': 128,
 'max_best_of': 2,
 'max_stop_sequences': 4,
 'max_input_length': 4096,
 'max_total_tokens': 6096,
 'waiting_served_ratio': 1.2,
 'max_batch_total_tokens': 75120,
 'max_waiting_tokens': 20,
 'max_batch_size': None,
 'validation_workers': 2,
 'version': '1.4.1',
 'sha': '4139054b82c7eedefcb401400d5ba4172a960ecc',
 'docker_label': 'sha-4139054'}

In [7]:
import json
s = r"""{"generated_text":"\n\nDeep Learning is a subset of Machine Learning that uses neural networks with multiple layers to process and","details":{"finish_reason":"length","generated_tokens":20,"seed":1,"prefill":[],"tokens":[{"id":13,"text":"\n","logprob":-0.0004518032,"special":false},{"id":13,"text":"\n","logprob":-0.0049209595,"special":false},{"id":23229,"text":"Deep","logprob":-0.00025963783,"special":false},{"id":17504,"text":" Learning","logprob":-1.2900391,"special":false},{"id":349,"text":" is","logprob":-0.00092840195,"special":false},{"id":264,"text":" a","logprob":-0.0031051636,"special":false},{"id":19804,"text":" subset","logprob":-0.07147217,"special":false},{"id":302,"text":" of","logprob":0.0,"special":false},{"id":13253,"text":" Machine","logprob":-0.31420898,"special":false},{"id":17504,"text":" Learning","logprob":-0.00028681755,"special":false},{"id":369,"text":" that","logprob":-2.3808594,"special":false},{"id":6098,"text":" uses","logprob":-0.13830566,"special":false},{"id":25726,"text":" neural","logprob":-0.47265625,"special":false},{"id":12167,"text":" networks","logprob":-0.00004541874,"special":false},{"id":395,"text":" with","logprob":-2.4414062,"special":false},{"id":5166,"text":" multiple","logprob":-0.045166016,"special":false},{"id":13083,"text":" layers","logprob":-0.00007021427,"special":false},{"id":298,"text":" to","logprob":-0.0052490234,"special":false},{"id":1759,"text":" process","logprob":-5.7382812,"special":false},{"id":304,"text":" and","logprob":-0.29418945,"special":false}]}}"""

json.loads(s)

{'generated_text': '\n\nDeep Learning is a subset of Machine Learning that uses neural networks with multiple layers to process and',
 'details': {'finish_reason': 'length',
  'generated_tokens': 20,
  'seed': 1,
  'prefill': [],
  'tokens': [{'id': 13,
    'text': '\n',
    'logprob': -0.0004518032,
    'special': False},
   {'id': 13, 'text': '\n', 'logprob': -0.0049209595, 'special': False},
   {'id': 23229, 'text': 'Deep', 'logprob': -0.00025963783, 'special': False},
   {'id': 17504, 'text': ' Learning', 'logprob': -1.2900391, 'special': False},
   {'id': 349, 'text': ' is', 'logprob': -0.00092840195, 'special': False},
   {'id': 264, 'text': ' a', 'logprob': -0.0031051636, 'special': False},
   {'id': 19804, 'text': ' subset', 'logprob': -0.07147217, 'special': False},
   {'id': 302, 'text': ' of', 'logprob': 0.0, 'special': False},
   {'id': 13253, 'text': ' Machine', 'logprob': -0.31420898, 'special': False},
   {'id': 17504,
    'text': ' Learning',
    'logprob': -0.000286817

In [4]:
tokenizer = load_tokenizer('teknium/OpenHermes-2.5-Mistral-7B')
chat_model = load_tgi_chat_model(
    inference_server_url="http://tgi:80",
    tokenizer=tokenizer
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [12]:
message = HumanMessage(content="What is Deep Learning?")
ai_message = chat_model.invoke([message])
content = ai_message.content
tokens = ai_message.additional_kwargs["token_texts"]
token_ids = ai_message.additional_kwargs["token_ids"]
token_logprobs = ai_message.additional_kwargs["token_logprobs"]

print(content)
print()
print(f"{tokens[:10]=}")
print(f"{token_ids[:10]=}")
print(f"{token_logprobs[:10]=}")

Deep Learning is a subfield of Machine Learning that uses Artificial Neural Networks, inspired by the structure and function of the human brain, to process and analyze large amounts of data. These networks are composed of multiple layers of interconnected nodes, or "neurons," that process information by performing mathematical operations. Deep Learning algorithms can automatically learn and extract features from raw data, allowing them to learn and improve over time. They are widely used in various applications, such as image and speech recognition, natural language processing, and autonomous systems.

tokens[:10]=['Deep', ' Learning', ' is', ' a', ' sub', 'field', ' of', ' Machine', ' Learning', ' that']
token_ids[:10]=[23229, 17504, 349, 264, 1083, 2222, 302, 13253, 17504, 369]
token_logprobs[:10]=[0.0, -0.3720703, 0.0, 0.0, -1.8701172, -0.13269043, 0.0, -0.46826172, 0.0, -1.9609375]
